# 📡 Telco Customer Churn — Standalone End-to-End Pipeline

A **fully self-contained** version of the production pipeline: every function and class
below is a verbatim copy of the code in `src/` and `main.py`, so this notebook runs
**without the project scripts** and — thanks to fixed random seeds throughout —
reproduces **exactly the same numbers, figures, and comparison table** as `python main.py`.

**Only requirements:** the Python packages in `requirements.txt` and an internet
connection for the first-time dataset download.

**Stages:** load → clean → EDA → feature engineering → split → feature-importance
analysis → preprocessing → training (3 models, tuned) → evaluation & comparison →
decision-threshold tuning → SHAP explainability.

In [ ]:
# Imports — everything the pipeline needs, nothing project-specific
import time
import logging
import warnings
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve,
    precision_recall_curve, average_precision_score,
)
import xgboost as xgb
import lightgbm as lgb
import shap

%matplotlib inline

# Same warning filters as main.py
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# The pipeline modules log through `logger`; a plain stdlib logger reproduces
# the same messages as notebook output (DEBUG suppressed, same as the scripts)
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s | %(message)s", force=True)
logger = logging.getLogger("churn-notebook")

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

## ⚙️ Configuration

The exact constants from `src/config.py`: seed, split fractions, feature lists, and the
hyperparameter search spaces. Paths anchor to the project root whether the notebook is
launched from `notebooks/` or the repository root.

In [ ]:
# Paths — anchored to the project root
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd

RAW_DATA_FILE = PROJECT_ROOT / "data" / "raw" / "Telco-Customer-Churn.csv"
FIGURES_DIR = PROJECT_ROOT / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
for _dir in [RAW_DATA_FILE.parent, FIGURES_DIR, REPORTS_DIR]:
    _dir.mkdir(parents=True, exist_ok=True)

# Dataset source (direct URL — no Kaggle credentials required)
DATASET_URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'

# Reproducibility & split fractions
RANDOM_STATE = 42
TEST_SIZE = 0.15       # 15% for test
VAL_SIZE = 0.176       # 15% of total = 17.6% of remaining 85%

# Target / identifier
TARGET = 'Churn'
ID_COLUMN = 'customerID'

# Feature definitions
BINARY_FEATURES = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
SPECIAL_BINARY = ['SeniorCitizen']   # already 0/1
MULTICLASS_FEATURES = ['MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaymentMethod']
NUMERIC_FEATURES = ['tenure', 'MonthlyCharges', 'TotalCharges']
SERVICE_COLUMNS = ['PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies']

# Engineered features (created by FeatureEngineer below)
ENGINEERED_NUMERIC = ['avg_monthly_charge', 'service_count', 'charge_tenure_ratio']
ENGINEERED_FLAGS = ['has_security_backup', 'high_value_short_tenure']
ENGINEERED_CATEGORICAL = ['tenure_group']
FLAG_FEATURES = SPECIAL_BINARY + ENGINEERED_FLAGS   # 0/1 columns passed through unscaled

# Hyperparameter search spaces
LOGISTIC_REGRESSION_PARAMS = {'classifier__C': [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
 'classifier__penalty': ['l2'],
 'classifier__solver': ['lbfgs', 'liblinear'],
 'classifier__max_iter': [1000]}

XGBOOST_PARAMS = {'classifier__n_estimators': [100, 200, 300, 500],
 'classifier__max_depth': [3, 4, 5, 6, 7],
 'classifier__learning_rate': [0.01, 0.05, 0.1, 0.2],
 'classifier__subsample': [0.7, 0.8, 0.9, 1.0],
 'classifier__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
 'classifier__min_child_weight': [1, 3, 5, 7],
 'classifier__gamma': [0, 0.1, 0.2, 0.3],
 'classifier__reg_alpha': [0, 0.01, 0.1, 1.0],
 'classifier__reg_lambda': [0.5, 1.0, 1.5, 2.0]}

LIGHTGBM_PARAMS = {'classifier__n_estimators': [100, 200, 300, 500],
 'classifier__max_depth': [3, 5, 7, -1],
 'classifier__learning_rate': [0.01, 0.05, 0.1, 0.2],
 'classifier__num_leaves': [15, 31, 50, 70],
 'classifier__subsample': [0.7, 0.8, 0.9, 1.0],
 'classifier__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
 'classifier__min_child_samples': [5, 10, 20, 30],
 'classifier__reg_alpha': [0, 0.01, 0.1, 1.0],
 'classifier__reg_lambda': [0.5, 1.0, 1.5, 2.0]}

# Training configuration
CV_FOLDS = 5
N_ITER_SEARCH = 50          # RandomizedSearchCV iterations
SCORING_METRIC = 'roc_auc'  # primary tuning metric

np.random.seed(RANDOM_STATE)   # same global seed as main.py

## 1️⃣ Data Loading & Validation

Downloads the IBM Telco dataset (7,043 customers × 21 columns) on first use, verifies
the expected schema, converts the known-bad `TotalCharges` column (blank strings for
brand-new customers) to numeric, and profiles the data — shape, missing values,
duplicates, and the class balance (~26.5% churners → **imbalanced problem**).

In [ ]:
# Expected columns in the raw dataset — used as a sanity check
EXPECTED_COLUMNS = ['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn']


def download_dataset(url: str = DATASET_URL, dest: Path = RAW_DATA_FILE) -> Path:
    """
    Download the dataset from a direct URL if not already present.

    Args:
        url: Direct download URL for the CSV file.
        dest: Local file path to save the downloaded CSV.

    Returns:
        Path to the downloaded (or existing) file.

    Raises:
        requests.HTTPError: If the download fails.
    """
    if dest.exists():
        logger.info(f"Dataset already exists at {dest} — skipping download.")
        return dest

    logger.info(f"Downloading dataset from {url}...")
    dest.parent.mkdir(parents=True, exist_ok=True)

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    dest.write_bytes(response.content)
    logger.info(f"Dataset saved to {dest} ({dest.stat().st_size / 1024:.1f} KB)")
    return dest

def load_raw_data(filepath: Path = RAW_DATA_FILE) -> pd.DataFrame:
    """
    Load the raw CSV dataset and perform initial type fixes.

    Known issue: 'TotalCharges' column contains blank strings (' ')
    for customers with tenure=0. These are converted to NaN here.

    Args:
        filepath: Path to the raw CSV file.

    Returns:
        DataFrame with basic type corrections applied.

    Raises:
        FileNotFoundError: If the file doesn't exist.
        ValueError: If the schema doesn't match expectations.
    """
    if not filepath.exists():
        raise FileNotFoundError(
            f"Dataset not found at {filepath}. Run download_dataset() first."
        )

    logger.info(f"Loading dataset from {filepath}...")
    df = pd.read_csv(filepath)

    # --- Schema validation ---
    missing_cols = set(EXPECTED_COLUMNS) - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing expected columns: {missing_cols}")

    # --- Fix TotalCharges: blank strings → NaN → float ---
    # 11 rows have ' ' (space) in TotalCharges — these are new customers
    # with tenure=0 who haven't been billed yet
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    logger.info(
        f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns"
    )
    return df

def get_data_profile(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Generate a comprehensive data profile for EDA.

    Returns a dictionary with:
    - Shape, dtypes, memory usage
    - Missing value counts and percentages
    - Duplicate row count
    - Class distribution of the target variable
    - Basic statistics

    Args:
        df: Input DataFrame.

    Returns:
        Dictionary containing profile metrics.
    """
    n_rows, n_cols = df.shape

    # Missing values
    missing = df.isnull().sum()
    missing_pct = (missing / n_rows * 100).round(2)
    missing_info = {
        col: {"count": int(missing[col]), "pct": float(missing_pct[col])}
        for col in df.columns if missing[col] > 0
    }

    # Duplicates (by customerID)
    n_duplicates = df.duplicated(subset=[ID_COLUMN]).sum() if ID_COLUMN in df.columns else 0

    # Class distribution
    if TARGET in df.columns:
        class_dist = df[TARGET].value_counts().to_dict()
        class_pct = df[TARGET].value_counts(normalize=True).mul(100).round(2).to_dict()
    else:
        class_dist = {}
        class_pct = {}

    profile = {
        "n_rows": n_rows,
        "n_cols": n_cols,
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
        "dtypes": df.dtypes.astype(str).to_dict(),
        "missing_values": missing_info,
        "n_missing_total": int(missing.sum()),
        "n_duplicates": int(n_duplicates),
        "class_distribution": class_dist,
        "class_percentage": class_pct,
    }

    logger.info(
        f"Profile: {n_rows} rows, {n_cols} cols, "
        f"{len(missing_info)} cols with missing values, "
        f"{n_duplicates} duplicate IDs"
    )

    return profile

In [ ]:
download_dataset()
df_raw = load_raw_data()
profile = get_data_profile(df_raw)

print(f"Class distribution: {profile['class_distribution']}")
df_raw.head()

## 2️⃣ Cleaning

Model-agnostic fixes applied before anything else:

- **Deduplicate by `customerID`** *while the ID is still present* — deduplicating on all
  columns after dropping the ID would delete distinct customers with identical profiles.
- **Drop `customerID`** — a unique identifier carries no signal.
- **`TotalCharges` blanks → 0.0** — the 11 blanks belong to `tenure = 0` customers who
  have not been billed yet.
- **Encode the target** — `Churn`: Yes → 1, No → 0.

In [ ]:
class DataCleaner(BaseEstimator, TransformerMixin):
    """
    Initial data cleaning steps applied before the main pipeline.

    Handles:
    - Removing duplicate customers (by customerID, while it still exists)
    - Dropping customerID (not a predictive feature)
    - Converting TotalCharges blanks to NaN → imputing with 0
      (these are new customers with tenure=0)
    - Encoding the target variable
    """

    def fit(self, X: pd.DataFrame, y: Optional[pd.Series] = None) -> "DataCleaner":
        """No fitting required."""
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Clean the DataFrame."""
        df = X.copy()

        # Remove duplicate customers by ID while the ID is still present.
        # (Deduplicating on all columns AFTER dropping the ID would delete
        # distinct customers who happen to share an identical profile.)
        if ID_COLUMN in df.columns:
            n_before = len(df)
            df = df.drop_duplicates(subset=[ID_COLUMN])
            n_dropped = n_before - len(df)
            if n_dropped > 0:
                logger.info(f"Dropped {n_dropped} duplicate customer IDs.")
            df = df.drop(columns=[ID_COLUMN])

        # Convert TotalCharges to numeric (blanks → NaN)
        if "TotalCharges" in df.columns:
            df["TotalCharges"] = pd.to_numeric(
                df["TotalCharges"], errors="coerce"
            )
            # Impute NaN with 0 — these are new customers (tenure=0)
            df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

        # Encode target variable
        if TARGET in df.columns and df[TARGET].dtype == object:
            df[TARGET] = df[TARGET].map({"Yes": 1, "No": 0})

        return df

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply initial cleaning to the raw DataFrame.

    This is a convenience function that wraps the DataCleaner transformer
    for use outside of the sklearn Pipeline (e.g., in EDA notebooks).

    Args:
        df: Raw DataFrame from data_loader.

    Returns:
        Cleaned DataFrame ready for feature engineering.
    """
    cleaner = DataCleaner()
    df_clean = cleaner.transform(df)
    logger.info(
        f"Cleaning complete: {df_clean.shape[0]} rows × {df_clean.shape[1]} cols"
    )
    return df_clean

In [ ]:
df_clean = clean_data(df_raw)
df_clean.head()

## 3️⃣ Exploratory Data Analysis

Ten charts saved to `figures/` — class balance, churn rate by contract / internet /
payment method / senior status / services, tenure & charge distributions by churn,
outlier box plots, and a correlation heatmap.

**Headline insights:** month-to-month contracts churn at ~43% vs ~3% for two-year;
fiber-optic customers churn ~2× DSL; the first 12 months are the riskiest; and
electronic-check payers churn far more than customers on automatic payments.

In [ ]:
def run_eda(df: pd.DataFrame) -> None:
    """
    Generate all EDA visualizations and save to figures/.

    Args:
        df: Cleaned DataFrame with target column.
    """
    logger.info("=" * 60)
    logger.info("EXPLORATORY DATA ANALYSIS")
    logger.info("=" * 60)

    plt.style.use("seaborn-v0_8-whitegrid")

    # 1. Churn Distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    churn_counts = df[TARGET].value_counts()
    colors = ["#2196F3", "#FF5722"]

    axes[0].bar(["No Churn", "Churn"], churn_counts.values, color=colors, edgecolor="white")
    axes[0].set_title("Churn Distribution", fontsize=13, fontweight="bold")
    axes[0].set_ylabel("Count")
    for i, v in enumerate(churn_counts.values):
        axes[0].text(i, v + 50, str(v), ha="center", fontweight="bold")

    axes[1].pie(
        churn_counts.values, labels=["No Churn", "Churn"],
        autopct="%1.1f%%", colors=colors, startangle=90,
        textprops={"fontsize": 12},
    )
    axes[1].set_title("Churn Percentage", fontsize=13, fontweight="bold")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_churn_distribution.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # 2. Churn by Contract Type
    if "Contract" in df.columns:
        fig, ax = plt.subplots(figsize=(8, 5))
        ct = pd.crosstab(df["Contract"], df[TARGET], normalize="index") * 100
        ct.columns = ["No Churn %", "Churn %"]
        ct.plot(kind="bar", stacked=True, color=colors, ax=ax, edgecolor="white")
        ax.set_title("Churn Rate by Contract Type", fontsize=13, fontweight="bold")
        ax.set_ylabel("Percentage (%)")
        ax.set_xlabel("")
        ax.legend(loc="upper right")
        plt.xticks(rotation=0)
        plt.tight_layout()
        fig.savefig(FIGURES_DIR / "eda_churn_by_contract.png", dpi=150, bbox_inches="tight")
        plt.close(fig)

    # 3. Churn by Internet Service
    if "InternetService" in df.columns:
        fig, ax = plt.subplots(figsize=(8, 5))
        ct = pd.crosstab(df["InternetService"], df[TARGET], normalize="index") * 100
        ct.columns = ["No Churn %", "Churn %"]
        ct.plot(kind="bar", stacked=True, color=colors, ax=ax, edgecolor="white")
        ax.set_title("Churn Rate by Internet Service", fontsize=13, fontweight="bold")
        ax.set_ylabel("Percentage (%)")
        ax.set_xlabel("")
        plt.xticks(rotation=0)
        plt.tight_layout()
        fig.savefig(FIGURES_DIR / "eda_churn_by_internet.png", dpi=150, bbox_inches="tight")
        plt.close(fig)

    # 4. Tenure Distribution by Churn
    fig, ax = plt.subplots(figsize=(10, 5))
    for churn_val, label, color in [(0, "No Churn", "#2196F3"), (1, "Churn", "#FF5722")]:
        subset = df[df[TARGET] == churn_val]["tenure"]
        ax.hist(subset, bins=30, alpha=0.6, label=label, color=color, edgecolor="white")
    ax.set_title("Tenure Distribution by Churn Status", fontsize=13, fontweight="bold")
    ax.set_xlabel("Tenure (months)")
    ax.set_ylabel("Count")
    ax.legend()
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_tenure_by_churn.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # 5. Monthly Charges Distribution by Churn
    fig, ax = plt.subplots(figsize=(10, 5))
    for churn_val, label, color in [(0, "No Churn", "#2196F3"), (1, "Churn", "#FF5722")]:
        subset = df[df[TARGET] == churn_val]["MonthlyCharges"]
        ax.hist(subset, bins=30, alpha=0.6, label=label, color=color, edgecolor="white")
    ax.set_title("Monthly Charges Distribution by Churn", fontsize=13, fontweight="bold")
    ax.set_xlabel("Monthly Charges ($)")
    ax.set_ylabel("Count")
    ax.legend()
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_monthly_charges_by_churn.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # 6. Correlation Heatmap
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 1:
        fig, ax = plt.subplots(figsize=(12, 10))
        corr = df[numeric_cols].corr()
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(
            corr, mask=mask, annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, ax=ax,
            square=True, linewidths=0.5,
            vmin=-1, vmax=1,
        )
        ax.set_title("Correlation Heatmap", fontsize=14, fontweight="bold")
        plt.tight_layout()
        fig.savefig(FIGURES_DIR / "eda_correlation_heatmap.png", dpi=150, bbox_inches="tight")
        plt.close(fig)

    # 7. Churn by Payment Method
    if "PaymentMethod" in df.columns:
        fig, ax = plt.subplots(figsize=(10, 5))
        ct = pd.crosstab(df["PaymentMethod"], df[TARGET], normalize="index") * 100
        ct.columns = ["No Churn %", "Churn %"]
        ct.plot(kind="bar", stacked=True, color=colors, ax=ax, edgecolor="white")
        ax.set_title("Churn Rate by Payment Method", fontsize=13, fontweight="bold")
        ax.set_ylabel("Percentage (%)")
        ax.set_xlabel("")
        plt.xticks(rotation=15)
        plt.tight_layout()
        fig.savefig(FIGURES_DIR / "eda_churn_by_payment.png", dpi=150, bbox_inches="tight")
        plt.close(fig)

    # 8. Box Plots for Outlier Detection
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for i, col in enumerate(["tenure", "MonthlyCharges", "TotalCharges"]):
        if col in df.columns:
            box_data = [df[df[TARGET] == 0][col], df[df[TARGET] == 1][col]]
            bp = axes[i].boxplot(box_data, tick_labels=["No Churn", "Churn"], patch_artist=True)
            bp["boxes"][0].set_facecolor("#2196F3")
            bp["boxes"][1].set_facecolor("#FF5722")
            axes[i].set_title(col, fontsize=12, fontweight="bold")
            axes[i].set_ylabel(col)
    plt.suptitle("Box Plots — Outlier Detection", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "eda_boxplots.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # 9. Churn by Senior Citizen
    if "SeniorCitizen" in df.columns:
        fig, ax = plt.subplots(figsize=(6, 5))
        ct = pd.crosstab(df["SeniorCitizen"], df[TARGET], normalize="index") * 100
        ct.columns = ["No Churn %", "Churn %"]
        ct.index = ["Non-Senior", "Senior"]
        ct.plot(kind="bar", stacked=True, color=colors, ax=ax, edgecolor="white")
        ax.set_title("Churn Rate by Senior Citizen Status", fontsize=13, fontweight="bold")
        ax.set_ylabel("Percentage (%)")
        ax.set_xlabel("")
        plt.xticks(rotation=0)
        plt.tight_layout()
        fig.savefig(FIGURES_DIR / "eda_churn_by_senior.png", dpi=150, bbox_inches="tight")
        plt.close(fig)

    # 10. Stacked Bar: Services vs Churn
    service_cols = ["OnlineSecurity", "TechSupport", "OnlineBackup",
                    "DeviceProtection", "StreamingTV", "StreamingMovies"]
    available_services = [c for c in service_cols if c in df.columns]
    if available_services:
        fig, ax = plt.subplots(figsize=(12, 6))
        churn_rates = {}
        for svc in available_services:
            rate = df.groupby(svc)[TARGET].mean() * 100
            for val, pct in rate.items():
                churn_rates[f"{svc}\n({val})"] = pct

        sorted_items = sorted(churn_rates.items(), key=lambda x: x[1], reverse=True)
        labels, values = zip(*sorted_items)
        bar_colors = ["#FF5722" if v > 30 else "#2196F3" for v in values]
        ax.barh(labels, values, color=bar_colors, edgecolor="white")
        ax.set_xlabel("Churn Rate (%)")
        ax.set_title("Churn Rate by Service Subscription", fontsize=13, fontweight="bold")
        ax.axvline(x=df[TARGET].mean() * 100, color="black", linestyle="--", alpha=0.5, label="Average")
        ax.legend()
        plt.tight_layout()
        fig.savefig(FIGURES_DIR / "eda_churn_by_services.png", dpi=150, bbox_inches="tight")
        plt.close(fig)

    logger.info(f"EDA complete. Saved {len(list(FIGURES_DIR.glob('eda_*.png')))} visualizations to {FIGURES_DIR}")

In [ ]:
run_eda(df_clean)

for fig_name in ["eda_churn_distribution.png", "eda_churn_by_contract.png",
                 "eda_tenure_by_churn.png", "eda_churn_by_services.png"]:
    display(Image(filename=str(FIGURES_DIR / fig_name)))

## 4️⃣ Feature Engineering

Six domain-driven features:

| Feature | Type | Rationale |
|---|---|---|
| `tenure_group` | categorical | Lifecycle bins (`0-12` … `61+`, open-ended → no NaN for future tenures) |
| `avg_monthly_charge` | numeric | `TotalCharges / max(tenure, 1)` — spending consistency |
| `service_count` | numeric | Active services (a service counts unless its value is a "No …" variant, so DSL/Fiber count as internet) — bundling raises switching costs |
| `has_security_backup` | flag | Has *both* OnlineSecurity and OnlineBackup |
| `high_value_short_tenure` | flag | Charges above the **training median** with tenure < 12 — the costliest churn segment |
| `charge_tenure_ratio` | numeric | Premium pricing relative to relationship length |

**Leakage / train-serve-skew guard:** the `FeatureEngineer` transformer runs *inside*
the model pipeline. Its `fit()` learns the `MonthlyCharges` median from the training
fold only and re-applies it identically to every future row — including single-row
predictions, where a per-DataFrame median would degenerate to the row's own value.

In [ ]:
def coerce_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Coerce the numeric base columns to numeric dtype.

    Inference data does not pass through DataCleaner (it is not part of the
    serving pipeline), so a raw CSV upload can deliver TotalCharges as
    strings with blanks for tenure-0 customers. This mirrors the cleaning
    semantics: blanks/garbage → NaN, TotalCharges NaN → 0 (new customers);
    remaining NaN is handled by the pipeline's SimpleImputer.
    """
    df = df.copy()
    for col in ("tenure", "MonthlyCharges", "TotalCharges"):
        if col in df.columns and df[col].dtype == object:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "TotalCharges" in df.columns:
        df["TotalCharges"] = df["TotalCharges"].fillna(0.0)
    return df

def create_tenure_group(df: pd.DataFrame) -> pd.DataFrame:
    """
    Bin tenure into lifecycle groups.

    Rationale: Customer behavior changes non-linearly with tenure.
    New customers (0-12 months) have the highest churn risk,
    while long-term customers (49+ months) are much more loyal.
    Binning captures these phase transitions.

    Groups:
    - 0-12:  New customer (highest churn risk)
    - 13-24: Early adopter
    - 25-36: Established
    - 37-48: Loyal
    - 49-60: Long-term
    - 61+:   Veteran (lowest churn risk)

    The last bin is open-ended so tenure values above 72 months
    (possible in future data) never produce NaN.
    """
    df = df.copy()
    bins = [0, 12, 24, 36, 48, 60, np.inf]
    labels = ["0-12", "13-24", "25-36", "37-48", "49-60", "61+"]
    df["tenure_group"] = pd.cut(
        df["tenure"], bins=bins, labels=labels, include_lowest=True
    ).astype(str)
    return df

def create_avg_monthly_charge(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate average monthly charge from total charges and tenure.

    Formula: TotalCharges / max(tenure, 1)

    Rationale: This measures spending consistency. A customer paying
    $70/month with a $70 average is stable, while one paying $70/month
    with a $50 average may have recently upgraded (potential satisfaction risk).

    The max(tenure, 1) prevents division by zero for brand-new customers.
    """
    df = df.copy()
    df["avg_monthly_charge"] = df["TotalCharges"] / df["tenure"].clip(lower=1)
    return df

def create_service_count(df: pd.DataFrame) -> pd.DataFrame:
    """
    Count the number of active services per customer.

    Rationale: Customers with more services have higher switching costs
    (they'd need to find multiple replacements). This is a strong
    predictor of retention — bundled customers are less likely to churn.

    Services counted: PhoneService, MultipleLines, InternetService,
    OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport,
    StreamingTV, StreamingMovies.

    A service is active unless its value is 'No' / 'No phone service' /
    'No internet service' (or missing). This rule is what lets
    InternetService count: its values are 'DSL' / 'Fiber optic' / 'No',
    never 'Yes', so a plain equals-'Yes' check would silently ignore the
    internet subscription for every internet customer.
    """
    df = df.copy()
    available_services = [col for col in SERVICE_COLUMNS if col in df.columns]

    if available_services:
        normalized = df[available_services].astype(str).apply(
            lambda col: col.str.strip().str.lower()
        )
        inactive = {"no", "no phone service", "no internet service", "nan", ""}
        df["service_count"] = (~normalized.isin(inactive)).sum(axis=1).astype(int)
    else:
        df["service_count"] = 0
    return df

def create_has_security_backup(df: pd.DataFrame) -> pd.DataFrame:
    """
    Flag customers who have BOTH OnlineSecurity AND OnlineBackup.

    Rationale: Customers with both protection services are typically
    more engaged and security-conscious. They've invested in protecting
    their data, which correlates with lower churn — they value the service.
    """
    df = df.copy()

    def is_yes(col: str) -> pd.Series:
        if col in df.columns:
            return df[col].astype(str).str.strip().str.lower() == "yes"
        # Missing column → False for every row (index-aligned, unlike df.get
        # with an empty-Series default, which silently produces NaN)
        return pd.Series(False, index=df.index)

    df["has_security_backup"] = (is_yes("OnlineSecurity") & is_yes("OnlineBackup")).astype(int)
    return df

def create_high_value_short_tenure(
    df: pd.DataFrame,
    charge_threshold: Optional[float] = None,
) -> pd.DataFrame:
    """
    Identify high-revenue customers who are still in the early lifecycle.

    Rationale: These are the most DANGEROUS churners — they spend above
    the median but haven't built loyalty yet (tenure < 12 months).
    Losing them has the highest revenue impact. This feature helps
    the model prioritize retention efforts on the most valuable at-risk segment.

    Args:
        df: Input DataFrame.
        charge_threshold: MonthlyCharges cutoff. Must come from TRAINING
            data statistics in production (see FeatureEngineer) — computing
            it from the scored data itself would make the feature meaningless
            for single-row predictions. If None, the median of `df` is used
            (acceptable only for EDA on the full dataset).
    """
    df = df.copy()
    if charge_threshold is None:
        charge_threshold = df["MonthlyCharges"].median()
    df["high_value_short_tenure"] = (
        (df["MonthlyCharges"] > charge_threshold) & (df["tenure"] < 12)
    ).astype(int)
    return df

def create_charge_tenure_ratio(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate the ratio of monthly charges to tenure.

    Rationale: High monthly charge relative to tenure suggests the
    customer is paying a premium without having built loyalty.
    This captures price sensitivity in the context of relationship duration.
    """
    df = df.copy()
    df["charge_tenure_ratio"] = df["MonthlyCharges"] / df["tenure"].clip(lower=1)
    return df

def engineer_features(
    df: pd.DataFrame,
    charge_threshold: Optional[float] = None,
) -> pd.DataFrame:
    """
    Apply all feature engineering transformations.

    For EDA and notebook use. Production code should use the
    FeatureEngineer transformer inside the model pipeline instead,
    so the MonthlyCharges threshold is learned from training data.

    Args:
        df: Cleaned DataFrame (after preprocessing.clean_data).
        charge_threshold: Optional fixed MonthlyCharges cutoff for
            high_value_short_tenure. Defaults to the median of `df`.

    Returns:
        DataFrame with all new features added.
    """
    df = create_tenure_group(df)
    df = create_avg_monthly_charge(df)
    df = create_service_count(df)
    df = create_has_security_backup(df)
    df = create_high_value_short_tenure(df, charge_threshold=charge_threshold)
    df = create_charge_tenure_ratio(df)

    new_features = ENGINEERED_CATEGORICAL + ENGINEERED_NUMERIC + ENGINEERED_FLAGS
    # DEBUG level: this runs on every pipeline transform (hundreds of times
    # during hyperparameter search), so INFO would flood the logs
    logger.debug(
        f"Feature engineering complete. Added {len(new_features)} features: "
        f"{sorted(new_features)}"
    )
    return df

class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Sklearn transformer that adds all engineered features.

    Designed to be the FIRST step of the model pipeline:

        Pipeline([
            ("features", FeatureEngineer()),
            ("preprocessor", ColumnTransformer(...)),
            ("classifier", ...),
        ])

    `fit` learns the MonthlyCharges median from the training fold only,
    so cross-validation is leakage-free and single-row API requests get
    the exact same feature definitions as training data.
    """

    def fit(self, X: pd.DataFrame, y=None) -> "FeatureEngineer":
        """Learn training-data statistics needed by the features."""
        if "MonthlyCharges" not in X.columns:
            raise ValueError("FeatureEngineer requires a 'MonthlyCharges' column.")
        X = coerce_numeric_columns(X)
        self.monthly_charges_median_ = float(X["MonthlyCharges"].median())
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Add all engineered features using statistics learned in fit."""
        X = coerce_numeric_columns(X)
        return engineer_features(X, charge_threshold=self.monthly_charges_median_)

    def get_feature_names_out(self, input_features=None) -> np.ndarray:
        """Return input feature names plus the engineered feature names."""
        if input_features is None:
            input_features = self.feature_names_in_
        # Same order in which transform() appends the new columns
        engineered = [
            "tenure_group", "avg_monthly_charge", "service_count",
            "has_security_backup", "high_value_short_tenure", "charge_tenure_ratio",
        ]
        new = [f for f in engineered if f not in list(input_features)]
        return np.concatenate([np.asarray(input_features, dtype=object), np.asarray(new, dtype=object)])

In [ ]:
# Exploration-only preview (median from the full data). During training the
# FeatureEngineer pipeline step recomputes everything per CV fold.
df_featured = engineer_features(df_clean)

new_cols = ["tenure_group", "avg_monthly_charge", "service_count",
            "has_security_backup", "high_value_short_tenure", "charge_tenure_ratio"]
df_featured[["tenure", "MonthlyCharges", "TotalCharges"] + new_cols].head(8)

## 5️⃣ Train / Validation / Test Split

Stratified 70 / 15 / 15 split on the **cleaned raw features** (feature engineering is
refit inside each model pipeline per fold). Roles: **train** — fitting + 5-fold CV
search; **validation** — model selection and threshold tuning; **test** — one final,
unbiased performance estimate.

In [ ]:
def split_data(
    df: pd.DataFrame,
    target: str = TARGET,
    test_size: float = TEST_SIZE,
    val_size: float = VAL_SIZE,
    random_state: int = RANDOM_STATE,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame,
           pd.Series, pd.Series, pd.Series]:
    """
    Split data into Train (70%) / Validation (15%) / Test (15%).

    Uses stratified splitting to preserve class distribution across all sets.
    The split is done in two steps:
    1. Split into train+val (85%) and test (15%)
    2. Split train+val into train (70%) and val (15%)

    Args:
        df: Cleaned DataFrame with target column.
        target: Target column name.
        test_size: Fraction reserved for test set.
        val_size: Fraction of train+val reserved for validation.
        random_state: Random seed for reproducibility.

    Returns:
        Tuple of (X_train, X_val, X_test, y_train, y_val, y_test)
    """
    X = df.drop(columns=[target])
    y = df[target]

    # Step 1: Split off test set
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )

    # Step 2: Split remaining into train and validation
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=val_size,
        stratify=y_temp,
        random_state=random_state,
    )

    logger.info(
        f"Data split: Train={len(X_train)} ({len(X_train)/len(df)*100:.1f}%), "
        f"Val={len(X_val)} ({len(X_val)/len(df)*100:.1f}%), "
        f"Test={len(X_test)} ({len(X_test)/len(df)*100:.1f}%)"
    )
    logger.info(
        f"Target distribution — "
        f"Train: {y_train.mean():.3f}, "
        f"Val: {y_val.mean():.3f}, "
        f"Test: {y_test.mean():.3f}"
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df_clean)

## 6️⃣ Preprocessing Architecture

A `ColumnTransformer` with four explicit branches:

| Branch | Columns | Transform |
|---|---|---|
| `num` | tenure, charges + engineered numerics | median-impute → `StandardScaler` |
| `bin` | Yes/No & gender columns | `BinaryEncoder` → 1/0 (fails fast on unmapped values) |
| `cat` | multi-class categoricals + `tenure_group` | `OneHotEncoder(drop=None, handle_unknown="ignore")` |
| `flag` | `SeniorCitizen` + engineered 0/1 flags | passthrough |

Safety choices: **`remainder="drop"`** ignores unexpected columns (`customerID`,
`Churn`, arbitrary CSV columns) instead of crashing; **`drop=None`** keeps unseen
categories distinguishable from any known category (with `drop="first"` an unknown
value would silently encode as the dropped baseline).

In [ ]:
class BinaryEncoder(BaseEstimator, TransformerMixin):
    """
    Encode binary Yes/No and Male/Female columns to 1/0.

    This is preferred over OneHotEncoder for binary features because:
    - It produces a single column (not two dummy columns)
    - It's more memory-efficient
    - It preserves interpretability

    Values are stripped of surrounding whitespace before mapping.
    Unmapped values raise a ValueError — failing fast beats silently
    feeding garbage into the model in production.
    """

    _MAPPING = {
        "Yes": 1, "No": 0,
        "Male": 1, "Female": 0,
    }

    def fit(self, X: pd.DataFrame, y: Optional[pd.Series] = None) -> "BinaryEncoder":
        """Record input feature names; the mapping itself is deterministic."""
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Apply binary encoding to all object-dtype columns."""
        X = X.copy()
        for col in X.columns:
            if X[col].dtype == object:
                stripped = X[col].astype(str).str.strip()
                mapped = stripped.map(self._MAPPING)
                unknown = mapped.isna() & X[col].notna()
                if unknown.any():
                    bad_values = sorted(stripped[unknown].unique().tolist())
                    raise ValueError(
                        f"BinaryEncoder: column '{col}' contains values that are "
                        f"not binary Yes/No or Male/Female: {bad_values}"
                    )
                n_missing = int(X[col].isna().sum())
                if n_missing:
                    logger.warning(
                        f"BinaryEncoder: column '{col}' has {n_missing} missing "
                        f"values; encoding them as 0 ('No')."
                    )
                X[col] = mapped.fillna(0).astype(int)
        return X

    def get_feature_names_out(self, input_features=None) -> np.ndarray:
        """Binary encoding is one-to-one: output names equal input names."""
        if input_features is not None:
            return np.asarray(input_features, dtype=object)
        return self.feature_names_in_

def build_preprocessing_pipeline(
    numeric_features: List[str],
    binary_features: List[str],
    multiclass_features: List[str],
    flag_features: Optional[List[str]] = None,
) -> ColumnTransformer:
    """
    Build a ColumnTransformer that handles all feature transformations.

    Architecture:
    - Numeric features → Impute missing → StandardScaler
    - Binary features → BinaryEncoder (Yes/No → 1/0)
    - Multi-class features → OneHotEncoder (drop first to avoid multicollinearity)
    - Flag features (already 0/1) → passthrough
    - Everything else → DROPPED. This is a deliberate safety net: extra
      columns in inference data (customerID, Churn, arbitrary CSV columns)
      are ignored instead of crashing or leaking into the model.

    This transformer is fit ONLY on training data to prevent data leakage.

    Args:
        numeric_features: List of numeric column names.
        binary_features: List of binary (Yes/No) column names.
        multiclass_features: List of multi-class categorical column names.
        flag_features: List of already-0/1 columns to pass through unchanged.

    Returns:
        Configured ColumnTransformer.
    """
    # Numeric pipeline: impute missing → scale
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    # Binary pipeline: encode Yes/No to 1/0
    binary_pipeline = Pipeline([
        ("encoder", BinaryEncoder()),
    ])

    # Multi-class pipeline: one-hot encode.
    # drop=None (not "first"): with drop="first" + handle_unknown="ignore",
    # an UNSEEN category at inference encodes as all-zeros — identical to the
    # dropped baseline category — so unknown values would silently score as
    # the baseline class. With drop=None, unknowns get a distinct all-zeros
    # row no real category shares. The regularized/tree models used here
    # don't need the collinearity drop.
    multiclass_pipeline = Pipeline([
        ("encoder", OneHotEncoder(
            drop=None,
            sparse_output=False,    # Return dense array for compatibility
            handle_unknown="ignore",  # Handle unseen categories gracefully in production
        )),
    ])

    transformers = [
        ("num", numeric_pipeline, numeric_features),
        ("bin", binary_pipeline, binary_features),
        ("cat", multiclass_pipeline, multiclass_features),
    ]
    if flag_features:
        transformers.append(("flag", "passthrough", flag_features))

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=True,
    )

    return preprocessor

def get_default_preprocessor() -> ColumnTransformer:
    """
    Build the default preprocessor for the full engineered feature set.

    Returns:
        ColumnTransformer covering original + engineered features.
    """
    return build_preprocessing_pipeline(
        numeric_features=NUMERIC_FEATURES + ENGINEERED_NUMERIC,
        binary_features=BINARY_FEATURES,
        multiclass_features=MULTICLASS_FEATURES + ENGINEERED_CATEGORICAL,
        flag_features=FLAG_FEATURES,
    )

In [ ]:
preprocessor = get_default_preprocessor()
preprocessor

## 7️⃣ Feature-Importance Analysis

Two complementary rankings on the fully transformed training matrix, after a
|r| > 0.9 correlation filter: **mutual information** (model-free, non-linear) and
**random forest importance** (interaction-aware). This is **analysis only** — the
models train on the full feature set; the rankings document where the signal lives.

In [ ]:
def correlation_filter(
    df: pd.DataFrame,
    threshold: float = 0.90,
    target: str = TARGET,
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Remove one feature from each pair of highly correlated features.

    When two features have |correlation| > threshold, the one with
    lower correlation to the target variable is dropped.

    Args:
        df: DataFrame with numeric features.
        threshold: Absolute correlation cutoff (default 0.90).
        target: Target column name.

    Returns:
        Tuple of (filtered DataFrame, list of dropped column names).
    """
    # Select only numeric columns for correlation
    numeric_df = df.select_dtypes(include=[np.number])

    if target in numeric_df.columns:
        numeric_for_corr = numeric_df.drop(columns=[target])
        target_corr = numeric_df.corrwith(numeric_df[target]).abs()
    else:
        numeric_for_corr = numeric_df
        target_corr = pd.Series(0, index=numeric_for_corr.columns)

    corr_matrix = numeric_for_corr.corr().abs()

    # Find pairs above threshold (upper triangle only to avoid double-counting)
    upper_tri = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    dropped_cols = []
    for col in upper_tri.columns:
        high_corr_cols = upper_tri.index[upper_tri[col] > threshold].tolist()
        for paired_col in high_corr_cols:
            # Drop the feature with lower target correlation
            if paired_col not in dropped_cols and col not in dropped_cols:
                if target_corr.get(col, 0) >= target_corr.get(paired_col, 0):
                    dropped_cols.append(paired_col)
                else:
                    dropped_cols.append(col)
                logger.info(
                    f"Correlation filter: dropping '{dropped_cols[-1]}' "
                    f"(r={corr_matrix.loc[col, paired_col]:.3f} with '{col if dropped_cols[-1] != col else paired_col}')"
                )

    df_filtered = df.drop(columns=[c for c in dropped_cols if c in df.columns])
    logger.info(
        f"Correlation filter: dropped {len(dropped_cols)} features. "
        f"Remaining: {df_filtered.shape[1]} columns."
    )
    return df_filtered, dropped_cols

def mutual_information_ranking(
    X: pd.DataFrame,
    y: pd.Series,
    top_n: int = 15,
) -> pd.DataFrame:
    """
    Rank features by Mutual Information with the target.

    Mutual Information measures the amount of information obtained about
    the target from observing each feature. It captures non-linear
    relationships that Pearson correlation misses.

    Args:
        X: Feature DataFrame (numeric only).
        y: Target Series.
        top_n: Number of top features to return.

    Returns:
        DataFrame with features ranked by MI score.
    """
    # Select only numeric features for MI calculation
    X_numeric = X.select_dtypes(include=[np.number])

    mi_scores = mutual_info_classif(
        X_numeric, y,
        random_state=RANDOM_STATE,
        n_neighbors=5,
    )

    mi_df = pd.DataFrame({
        "feature": X_numeric.columns,
        "mi_score": mi_scores,
    }).sort_values("mi_score", ascending=False).reset_index(drop=True)

    logger.info(f"Top {top_n} features by Mutual Information:")
    for _, row in mi_df.head(top_n).iterrows():
        logger.info(f"  {row['feature']}: {row['mi_score']:.4f}")

    # Save visualization
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.barplot(
        data=mi_df.head(top_n),
        x="mi_score", y="feature",
        hue="feature",
        palette="viridis",
        legend=False,
        ax=ax,
    )
    ax.set_title("Feature Ranking — Mutual Information", fontsize=14, fontweight="bold")
    ax.set_xlabel("Mutual Information Score")
    ax.set_ylabel("")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "feature_selection_mi.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    return mi_df

def model_based_importance(
    X: pd.DataFrame,
    y: pd.Series,
    top_n: int = 15,
) -> pd.DataFrame:
    """
    Rank features using Random Forest feature importances.

    Random Forest importance measures how much each feature contributes
    to reducing impurity (Gini) across all trees. This is a good
    complement to MI because it accounts for feature interactions.

    Args:
        X: Feature DataFrame (numeric only).
        y: Target Series.
        top_n: Number of top features to return.

    Returns:
        DataFrame with features ranked by importance.
    """
    X_numeric = X.select_dtypes(include=[np.number])

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced",
    )
    rf.fit(X_numeric, y)

    importance_df = pd.DataFrame({
        "feature": X_numeric.columns,
        "importance": rf.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    logger.info(f"Top {top_n} features by Random Forest importance:")
    for _, row in importance_df.head(top_n).iterrows():
        logger.info(f"  {row['feature']}: {row['importance']:.4f}")

    # Save visualization
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.barplot(
        data=importance_df.head(top_n),
        x="importance", y="feature",
        hue="feature",
        palette="magma",
        legend=False,
        ax=ax,
    )
    ax.set_title("Feature Ranking — Random Forest Importance", fontsize=14, fontweight="bold")
    ax.set_xlabel("Feature Importance (Gini)")
    ax.set_ylabel("")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "feature_selection_rf.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    return importance_df

def select_features(
    X: pd.DataFrame,
    y: pd.Series,
    top_n: int = 15,
    corr_threshold: float = 0.90,
) -> Tuple[List[str], Dict[str, pd.DataFrame]]:
    """
    Run the full feature selection pipeline.

    Strategy:
    1. Apply correlation filter to remove redundant features
    2. Rank remaining features with MI and RF importance
    3. Take the union of top features from both methods

    Args:
        X: Feature DataFrame.
        y: Target Series.
        top_n: Number of top features from each method.
        corr_threshold: Correlation threshold for filtering.

    Returns:
        Tuple of (selected feature names, dict of ranking DataFrames).
    """
    logger.info("=" * 60)
    logger.info("FEATURE SELECTION PIPELINE")
    logger.info("=" * 60)

    # Step 1: Correlation filter
    X_filtered, dropped = correlation_filter(
        X.assign(**{TARGET: y}), threshold=corr_threshold, target=TARGET,
    )
    if TARGET in X_filtered.columns:
        X_filtered = X_filtered.drop(columns=[TARGET])

    # Step 2: Mutual Information ranking
    mi_df = mutual_information_ranking(X_filtered, y, top_n=top_n)

    # Step 3: Model-based importance ranking
    rf_df = model_based_importance(X_filtered, y, top_n=top_n)

    # Step 4: Union of top features from both methods
    mi_top = set(mi_df.head(top_n)["feature"].tolist())
    rf_top = set(rf_df.head(top_n)["feature"].tolist())
    selected = sorted(mi_top | rf_top)

    # Only keep features that exist in X_filtered
    selected = [f for f in selected if f in X_filtered.columns]

    logger.info(f"Selected {len(selected)} features (union of MI + RF top-{top_n}):")
    for feat in selected:
        logger.info(f"  ✓ {feat}")

    rankings = {"mutual_information": mi_df, "random_forest": rf_df}
    return selected, rankings

In [ ]:
# Transform the training data exactly the way the models will see it
analysis_pipeline = Pipeline([
    ("features", FeatureEngineer()),          # median learned from X_train only
    ("preprocessor", get_default_preprocessor()),
])
X_train_transformed = analysis_pipeline.fit_transform(X_train, y_train)
feature_names_out = list(analysis_pipeline.named_steps["preprocessor"].get_feature_names_out())
X_train_df = pd.DataFrame(X_train_transformed, columns=feature_names_out)

selected_features, rankings = select_features(X_train_df, y_train.reset_index(drop=True), top_n=15)

display(Image(filename=str(FIGURES_DIR / "feature_selection_mi.png")))
display(Image(filename=str(FIGURES_DIR / "feature_selection_rf.png")))

## 8️⃣ Model Training & Hyperparameter Tuning

Three candidates, each a self-contained pipeline `FeatureEngineer → ColumnTransformer
→ classifier`, tuned with `RandomizedSearchCV` (50 candidates × 5 stratified folds,
optimizing ROC-AUC). Class imbalance is handled natively: balanced class weights (LR),
`scale_pos_weight` ≈ 2.8 (XGBoost), `is_unbalance` (LightGBM).

> ⚙️ Classifiers run single-threaded (`n_jobs=1`) while the search parallelizes across
> processes — nested thread pools oversubscribe cores and can crash loky workers on
> Windows. Expect ~40–60s total on a modern machine.

In [ ]:
def get_model_configs(
    scale_pos_weight: float = 1.0,
) -> Dict[str, Dict[str, Any]]:
    """
    Define all model configurations with their classifiers and param grids.

    Args:
        scale_pos_weight: Ratio of negative to positive class for
                          imbalanced dataset handling. Calculated as
                          n_negative / n_positive.

    Returns:
        Dictionary mapping model names to their config dicts containing:
        - 'classifier': sklearn-compatible classifier instance
        - 'params': hyperparameter search space
        - 'description': human-readable model description
    """
    configs = {
        "Logistic Regression": {
            "classifier": LogisticRegression(
                random_state=RANDOM_STATE,
                class_weight="balanced",  # Handles class imbalance
                max_iter=1000,
            ),
            "params": LOGISTIC_REGRESSION_PARAMS,
            "description": (
                "Interpretable linear baseline. Uses L2 regularization "
                "and balanced class weights to handle imbalanced data."
            ),
        },
        "XGBoost": {
            "classifier": xgb.XGBClassifier(
                random_state=RANDOM_STATE,
                scale_pos_weight=scale_pos_weight,
                eval_metric="logloss",
                verbosity=0,
                # Single-threaded: RandomizedSearchCV already parallelizes across
                # processes; nested thread pools oversubscribe cores and can get
                # loky workers OOM-killed on Windows.
                n_jobs=1,
            ),
            "params": XGBOOST_PARAMS,
            "description": (
                "Gradient boosting with regularization. Uses scale_pos_weight "
                "for class imbalance and L1/L2 regularization to prevent overfitting."
            ),
        },
        "LightGBM": {
            "classifier": lgb.LGBMClassifier(
                random_state=RANDOM_STATE,
                is_unbalance=True,  # Handles class imbalance
                verbosity=-1,
                n_jobs=1,  # See XGBoost note: search-level parallelism only
            ),
            "params": LIGHTGBM_PARAMS,
            "description": (
                "Fast gradient boosting with leaf-wise growth. Uses is_unbalance "
                "for class imbalance and histogram-based splitting for speed."
            ),
        },
    }

    return configs

def build_model_pipeline(
    preprocessor: ColumnTransformer,
    classifier: Any,
) -> Pipeline:
    """
    Create an sklearn Pipeline: feature engineering → preprocessing → classifier.

    The pipeline ensures that:
    - Engineered-feature statistics (MonthlyCharges median) and all
      preprocessing are fit ONLY on training data
    - Cross-validation properly re-fits every step per fold
    - The serialized pipeline is fully self-contained for deployment:
      it accepts raw customer records, no manual feature engineering needed

    Args:
        preprocessor: Unfitted ColumnTransformer (cloned per model).
        classifier: sklearn-compatible classifier instance.

    Returns:
        sklearn Pipeline.
    """
    return Pipeline([
        ("features", FeatureEngineer()),
        ("preprocessor", clone(preprocessor)),
        ("classifier", classifier),
    ])

def train_single_model(
    pipeline: Pipeline,
    param_grid: Dict[str, Any],
    X_train: pd.DataFrame,
    y_train: pd.Series,
    cv_folds: int = CV_FOLDS,
    n_iter: int = N_ITER_SEARCH,
    scoring: str = SCORING_METRIC,
) -> Tuple[Pipeline, Dict[str, Any], float]:
    """
    Train a single model with hyperparameter tuning via RandomizedSearchCV.

    Uses stratified K-fold cross-validation to ensure each fold has
    the same class distribution as the full training set.

    Args:
        pipeline: sklearn Pipeline with preprocessor + classifier.
        param_grid: Hyperparameter search space.
        X_train: Training features.
        y_train: Training labels.
        cv_folds: Number of CV folds.
        n_iter: Number of random parameter combinations to try.
        scoring: Metric to optimize.

    Returns:
        Tuple of (best pipeline, best params, training time in seconds).
    """
    cv = StratifiedKFold(
        n_splits=cv_folds,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_grid,
        n_iter=min(n_iter, _count_param_combinations(param_grid)),
        cv=cv,
        scoring=scoring,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
        return_train_score=True,
    )

    start_time = time.time()
    search.fit(X_train, y_train)
    train_time = time.time() - start_time

    best_params = search.best_params_
    best_score = search.best_score_

    logger.info(
        f"Best CV {scoring}: {best_score:.4f} "
        f"(training time: {train_time:.1f}s)"
    )
    logger.info(f"Best params: {best_params}")

    return search.best_estimator_, best_params, train_time

def _count_param_combinations(param_grid: Dict[str, Any]) -> int:
    """Count the total number of parameter combinations in a grid."""
    total = 1
    for values in param_grid.values():
        if isinstance(values, list):
            total *= len(values)
    return total

def train_all_models(
    preprocessor: ColumnTransformer,
    X_train: pd.DataFrame,
    y_train: pd.Series,
) -> Dict[str, Dict[str, Any]]:
    """
    Train all models with hyperparameter tuning.

    This is the main entry point for model training. It:
    1. Calculates class imbalance ratio
    2. Builds pipelines for each model
    3. Runs RandomizedSearchCV for each
    4. Returns all results in a structured dictionary

    Args:
        preprocessor: ColumnTransformer for feature transformation.
        X_train: Training features.
        y_train: Training labels.

    Returns:
        Dictionary mapping model names to result dicts containing:
        - 'pipeline': Best fitted pipeline
        - 'best_params': Best hyperparameters
        - 'train_time': Training time in seconds
        - 'description': Model description
    """
    # Calculate class imbalance ratio for XGBoost
    n_negative = (y_train == 0).sum()
    n_positive = (y_train == 1).sum()
    scale_pos_weight = n_negative / n_positive
    logger.info(
        f"Class imbalance: {n_negative} negative / {n_positive} positive "
        f"(ratio: {scale_pos_weight:.2f})"
    )

    model_configs = get_model_configs(scale_pos_weight=scale_pos_weight)
    results = {}

    for name, config in model_configs.items():
        logger.info("=" * 60)
        logger.info(f"Training: {name}")
        logger.info("=" * 60)

        pipeline = build_model_pipeline(
            preprocessor=preprocessor,
            classifier=config["classifier"],
        )

        best_pipeline, best_params, train_time = train_single_model(
            pipeline=pipeline,
            param_grid=config["params"],
            X_train=X_train,
            y_train=y_train,
        )

        results[name] = {
            "pipeline": best_pipeline,
            "best_params": best_params,
            "train_time": train_time,
            "description": config["description"],
        }

    logger.info("=" * 60)
    logger.info("All models trained successfully.")
    logger.info("=" * 60)

    return results

In [ ]:
model_results = train_all_models(preprocessor, X_train, y_train)

## 9️⃣ Evaluation, Comparison & Model Selection

Accuracy, precision, recall, F1 and ROC-AUC on **all three splits**, confusion
matrices, overlaid ROC / precision-recall curves, and an overfitting check
(train-vs-validation AUC gap).

**The winner is chosen by *validation* ROC-AUC** — selecting on the test set would
leak test information into model choice. All three models typically land within
~0.002 AUC of each other here (test ROC-AUC ≈ 0.85).

*A note on accuracy:* with 26.5% churners, always predicting "no churn" already scores
73.5%. The models trade accuracy for **recall** (~78–84% of churners caught at the 0.5
cutoff) because missing a churner costs far more than a false alarm.

In [ ]:
def compute_metrics(
    y_true: pd.Series,
    y_pred: np.ndarray,
    y_prob: np.ndarray,
    dataset_name: str = "Test",
) -> Dict[str, float]:
    """
    Compute all classification metrics for a single dataset split.

    Args:
        y_true: Ground truth labels.
        y_pred: Predicted binary labels.
        y_prob: Predicted probabilities for the positive class.
        dataset_name: Name of the dataset split (for logging).

    Returns:
        Dictionary of metric name → value.
    """
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
    }

    logger.info(
        f"{dataset_name} metrics — "
        f"Acc: {metrics['accuracy']:.4f}, "
        f"Prec: {metrics['precision']:.4f}, "
        f"Rec: {metrics['recall']:.4f}, "
        f"F1: {metrics['f1']:.4f}, "
        f"AUC: {metrics['roc_auc']:.4f}"
    )

    return metrics

def evaluate_model(
    pipeline: Pipeline,
    X_train: pd.DataFrame, y_train: pd.Series,
    X_val: pd.DataFrame, y_val: pd.Series,
    X_test: pd.DataFrame, y_test: pd.Series,
    model_name: str,
) -> Dict[str, Dict[str, float]]:
    """
    Evaluate a single model on all three dataset splits.

    Args:
        pipeline: Fitted sklearn Pipeline.
        X_train, y_train: Training data.
        X_val, y_val: Validation data.
        X_test, y_test: Test data.
        model_name: Name for logging and labeling.

    Returns:
        Nested dict: {split_name: {metric_name: value}}.
    """
    logger.info(f"\n--- Evaluating: {model_name} ---")
    results = {}

    for split_name, X, y in [
        ("train", X_train, y_train),
        ("validation", X_val, y_val),
        ("test", X_test, y_test),
    ]:
        y_pred = pipeline.predict(X)
        y_prob = pipeline.predict_proba(X)[:, 1]
        results[split_name] = compute_metrics(y, y_pred, y_prob, split_name.title())

    # Overfitting check: train vs validation gap
    train_auc = results["train"]["roc_auc"]
    val_auc = results["validation"]["roc_auc"]
    gap = train_auc - val_auc
    if gap > 0.05:
        logger.warning(
            f"⚠ Possible overfitting: Train AUC ({train_auc:.4f}) - "
            f"Val AUC ({val_auc:.4f}) = {gap:.4f}"
        )
    else:
        logger.info(
            f"✓ No significant overfitting: "
            f"Train-Val AUC gap = {gap:.4f}"
        )

    return results

def plot_confusion_matrix(
    pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    model_name: str,
    dataset_name: str = "Test",
) -> None:
    """
    Plot a confusion matrix heatmap.

    Args:
        pipeline: Fitted Pipeline.
        X: Features.
        y: True labels.
        model_name: Model name for the title.
        dataset_name: Dataset split name.
    """
    y_pred = pipeline.predict(X)
    cm = confusion_matrix(y, y_pred)

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["No Churn", "Churn"],
        yticklabels=["No Churn", "Churn"],
        ax=ax, cbar=False,
        annot_kws={"size": 14},
    )
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("Actual", fontsize=12)
    ax.set_title(
        f"Confusion Matrix — {model_name} ({dataset_name})",
        fontsize=13, fontweight="bold",
    )
    plt.tight_layout()

    filename = f"confusion_matrix_{model_name.lower().replace(' ', '_')}_{dataset_name.lower()}.png"
    fig.savefig(FIGURES_DIR / filename, dpi=150, bbox_inches="tight")
    plt.close(fig)

def plot_roc_curves(
    model_results: Dict[str, Dict[str, Any]],
    X_test: pd.DataFrame,
    y_test: pd.Series,
) -> None:
    """
    Plot ROC curves for all models on the test set (overlay).

    This allows direct visual comparison of discriminative ability.
    """
    fig, ax = plt.subplots(figsize=(8, 6))

    colors = ["#2196F3", "#FF5722", "#4CAF50", "#9C27B0", "#FF9800"]

    for idx, (name, result) in enumerate(model_results.items()):
        pipeline = result["pipeline"]
        y_prob = pipeline.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)

        ax.plot(
            fpr, tpr,
            label=f"{name} (AUC = {auc:.4f})",
            color=colors[idx % len(colors)],
            linewidth=2,
        )

    # Diagonal reference line (random classifier)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random (AUC = 0.5)")

    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title("ROC Curves — Model Comparison", fontsize=14, fontweight="bold")
    ax.legend(loc="lower right", fontsize=10)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])
    plt.tight_layout()

    fig.savefig(FIGURES_DIR / "roc_curves_comparison.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    logger.info("Saved ROC curves comparison plot.")

def plot_precision_recall_curves(
    model_results: Dict[str, Dict[str, Any]],
    X_test: pd.DataFrame,
    y_test: pd.Series,
) -> None:
    """
    Plot Precision-Recall curves for all models on the test set.

    PR curves are more informative than ROC when classes are imbalanced
    because they focus on the performance for the minority class.
    """
    fig, ax = plt.subplots(figsize=(8, 6))

    colors = ["#2196F3", "#FF5722", "#4CAF50", "#9C27B0", "#FF9800"]

    for idx, (name, result) in enumerate(model_results.items()):
        pipeline = result["pipeline"]
        y_prob = pipeline.predict_proba(X_test)[:, 1]
        precision, recall, _ = precision_recall_curve(y_test, y_prob)
        ap = average_precision_score(y_test, y_prob)

        ax.plot(
            recall, precision,
            label=f"{name} (AP = {ap:.4f})",
            color=colors[idx % len(colors)],
            linewidth=2,
        )

    # Baseline: prevalence of positive class
    baseline = y_test.mean()
    ax.axhline(y=baseline, color="k", linestyle="--", alpha=0.5, label=f"Baseline ({baseline:.3f})")

    ax.set_xlabel("Recall", fontsize=12)
    ax.set_ylabel("Precision", fontsize=12)
    ax.set_title("Precision-Recall Curves — Model Comparison", fontsize=14, fontweight="bold")
    ax.legend(loc="upper right", fontsize=10)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])
    plt.tight_layout()

    fig.savefig(FIGURES_DIR / "pr_curves_comparison.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    logger.info("Saved Precision-Recall curves comparison plot.")

def create_comparison_table(
    all_metrics: Dict[str, Dict[str, Dict[str, float]]],
    training_times: Dict[str, float],
    descriptions: Dict[str, str],
) -> pd.DataFrame:
    """
    Create a comprehensive model comparison table.

    Columns include performance metrics on all splits, training time,
    and overfitting gap (train AUC - validation AUC).

    Args:
        all_metrics: {model_name: {split: {metric: value}}}.
        training_times: {model_name: seconds}.
        descriptions: {model_name: description string}.

    Returns:
        DataFrame with one row per model, sorted by validation ROC-AUC
        (the selection metric; test columns are reported but never used
        to pick the winner).
    """
    rows = []
    for name in all_metrics:
        test = all_metrics[name].get("test", {})
        val = all_metrics[name].get("validation", {})
        train = all_metrics[name].get("train", {})

        row = {
            "Model": name,
            "Test Accuracy": test.get("accuracy", 0),
            "Test Precision": test.get("precision", 0),
            "Test Recall": test.get("recall", 0),
            "Test F1": test.get("f1", 0),
            "Test ROC-AUC": test.get("roc_auc", 0),
            "Val ROC-AUC": val.get("roc_auc", 0),
            "Train ROC-AUC": train.get("roc_auc", 0),
            "Overfit Gap": train.get("roc_auc", 0) - val.get("roc_auc", 0),
            "Training Time (s)": training_times.get(name, 0),
            "Description": descriptions.get(name, ""),
        }
        rows.append(row)

    comparison_df = pd.DataFrame(rows)
    # Rank by VALIDATION ROC-AUC: choosing the winner on the test set would
    # leak test information into model selection and bias the reported score.
    comparison_df = comparison_df.sort_values("Val ROC-AUC", ascending=False)
    comparison_df = comparison_df.reset_index(drop=True)

    # Save to CSV
    comparison_df.to_csv(REPORTS_DIR / "model_comparison.csv", index=False)
    logger.info(f"Model comparison table saved to {REPORTS_DIR / 'model_comparison.csv'}")

    # Pretty print
    display_cols = [
        "Model", "Test Accuracy", "Test Precision", "Test Recall",
        "Test F1", "Test ROC-AUC", "Val ROC-AUC", "Overfit Gap", "Training Time (s)",
    ]
    logger.info("\n" + comparison_df[display_cols].to_string(index=False))

    return comparison_df

def plot_comparison_bar_chart(comparison_df: pd.DataFrame) -> None:
    """
    Create a grouped bar chart comparing key metrics across models.
    """
    metrics = ["Test Accuracy", "Test Precision", "Test Recall", "Test F1", "Test ROC-AUC"]
    models = comparison_df["Model"].tolist()

    fig, ax = plt.subplots(figsize=(12, 6))

    x = np.arange(len(metrics))
    width = 0.25
    offsets = np.linspace(-width, width, len(models))

    colors = ["#2196F3", "#FF5722", "#4CAF50"]
    for idx, model in enumerate(models):
        values = comparison_df[comparison_df["Model"] == model][metrics].values[0]
        ax.bar(
            x + offsets[idx], values,
            width=width, label=model,
            color=colors[idx % len(colors)],
            alpha=0.85, edgecolor="white",
        )

    ax.set_xticks(x)
    ax.set_xticklabels([m.replace("Test ", "") for m in metrics], fontsize=11)
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title("Model Performance Comparison", fontsize=14, fontweight="bold")
    ax.legend(fontsize=10)
    ax.set_ylim([0, 1.05])
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    fig.savefig(FIGURES_DIR / "model_comparison_bar.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    logger.info("Saved model comparison bar chart.")

def evaluate_all_models(
    model_results: Dict[str, Dict[str, Any]],
    X_train: pd.DataFrame, y_train: pd.Series,
    X_val: pd.DataFrame, y_val: pd.Series,
    X_test: pd.DataFrame, y_test: pd.Series,
) -> pd.DataFrame:
    """
    Run the complete evaluation pipeline for all trained models.

    Steps:
    1. Compute metrics on all splits for each model
    2. Generate confusion matrices
    3. Plot ROC and PR curves
    4. Create comparison table and bar chart

    Args:
        model_results: Output from model_training.train_all_models().
        X_train, y_train, X_val, y_val, X_test, y_test: Data splits.

    Returns:
        Comparison DataFrame.
    """
    logger.info("=" * 60)
    logger.info("MODEL EVALUATION")
    logger.info("=" * 60)

    all_metrics = {}
    training_times = {}
    descriptions = {}

    for name, result in model_results.items():
        pipeline = result["pipeline"]

        # Compute metrics on all splits
        metrics = evaluate_model(
            pipeline,
            X_train, y_train,
            X_val, y_val,
            X_test, y_test,
            model_name=name,
        )
        all_metrics[name] = metrics
        training_times[name] = result["train_time"]
        descriptions[name] = result["description"]

        # Plot confusion matrices (test set)
        plot_confusion_matrix(pipeline, X_test, y_test, name, "Test")

    # Multi-model comparison plots
    plot_roc_curves(model_results, X_test, y_test)
    plot_precision_recall_curves(model_results, X_test, y_test)

    # Comparison table
    comparison_df = create_comparison_table(all_metrics, training_times, descriptions)
    plot_comparison_bar_chart(comparison_df)

    # Select best model
    best_model_name = comparison_df.iloc[0]["Model"]
    logger.info(f"\n🏆 Best model: {best_model_name} (highest Validation ROC-AUC)")

    return comparison_df

In [ ]:
comparison_df = evaluate_all_models(
    model_results, X_train, y_train, X_val, y_val, X_test, y_test,
)

best_model_name = comparison_df.iloc[0]["Model"]   # ranked by Val ROC-AUC
best_pipeline = model_results[best_model_name]["pipeline"]
print(f"Best model (by validation ROC-AUC): {best_model_name}")

comparison_df.drop(columns=["Description"])

In [ ]:
display(Image(filename=str(FIGURES_DIR / "roc_curves_comparison.png")))
display(Image(filename=str(FIGURES_DIR / "pr_curves_comparison.png")))
display(Image(filename=str(FIGURES_DIR / "model_comparison_bar.png")))

### Decision-Threshold Tuning

A 0.5 cutoff is rarely optimal for imbalanced data — and balanced class weights shift
predicted probabilities upward. The threshold that maximizes F1 is found on the
**validation set** (≈ 0.67 for the balanced logistic regression) and defines the
deployed operating point reported below.

In [ ]:
def tune_decision_threshold(
    pipeline: Pipeline,
    X_val: pd.DataFrame,
    y_val: pd.Series,
) -> float:
    """
    Find the probability cutoff that maximizes F1 on the validation set.

    The default 0.5 cutoff is rarely optimal for imbalanced problems —
    with ~27% churners, a lower threshold typically trades a little
    precision for substantially better recall. The tuned value is saved
    with the model metadata and applied by the API at serving time.

    Args:
        pipeline: Fitted pipeline of the selected best model.
        X_val: Validation features (raw).
        y_val: Validation labels.

    Returns:
        Optimal threshold as a float in (0, 1).
    """
    probs = pipeline.predict_proba(X_val)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y_val, probs)

    # precision/recall have one more element than thresholds; align them
    precision, recall = precision[:-1], recall[:-1]
    denom = precision + recall
    f1_scores = np.divide(
        2 * precision * recall, denom,
        out=np.zeros_like(denom), where=denom > 0,
    )
    best_idx = int(np.argmax(f1_scores))
    best_threshold = float(thresholds[best_idx])

    logger.info(
        f"Tuned decision threshold on validation set: {best_threshold:.3f} "
        f"(F1={f1_scores[best_idx]:.4f}, "
        f"precision={precision[best_idx]:.4f}, recall={recall[best_idx]:.4f})"
    )
    return best_threshold

In [ ]:
decision_threshold = tune_decision_threshold(best_pipeline, X_val, y_val)

# Final report: test metrics at the DEPLOYED operating point
test_probs = best_pipeline.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= decision_threshold).astype(int)

model_summary = {
    "best_model": best_model_name,
    "decision_threshold": round(decision_threshold, 4),
    "val_roc_auc": round(float(comparison_df.iloc[0]["Val ROC-AUC"]), 4),
    "test_roc_auc": round(float(comparison_df.iloc[0]["Test ROC-AUC"]), 4),
    "test_metrics_at_threshold": {
        "accuracy": round(float(accuracy_score(y_test, test_preds)), 4),
        "precision": round(float(precision_score(y_test, test_preds, zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, test_preds, zero_division=0)), 4),
        "f1": round(float(f1_score(y_test, test_preds, zero_division=0)), 4),
    },
}
model_summary

## 🔟 SHAP Explainability

Game-theoretic feature attributions with the right explainer per model type:
`TreeExplainer` for XGBoost/LightGBM, the exact-and-fast `LinearExplainer` for logistic
regression (`KernelExplainer` would need ~2,000 model evaluations *per row*).

Outputs: beeswarm summary, global importance bar chart, single-prediction waterfall,
and dependence plots for the top-3 features. Consistent top drivers: contract type,
tenure, fiber-optic internet, monthly charges, and electronic-check payment.

In [ ]:
def transform_features(pipeline: Pipeline, X: pd.DataFrame) -> pd.DataFrame:
    """
    Run raw features through every pipeline step except the final classifier
    and return the result as a DataFrame with proper feature names.

    Args:
        pipeline: Fitted Pipeline (feature engineering + preprocessing + classifier).
        X: Raw input features.

    Returns:
        Transformed DataFrame in model-input space.
    """
    transformer = Pipeline(pipeline.steps[:-1])
    X_transformed = transformer.transform(X)
    if hasattr(X_transformed, "toarray"):
        X_transformed = X_transformed.toarray()

    preprocessor = pipeline.named_steps["preprocessor"]
    try:
        feature_names = list(preprocessor.get_feature_names_out())
    except Exception:
        logger.warning("Could not extract feature names from preprocessor.")
        feature_names = [f"feature_{i}" for i in range(X_transformed.shape[1])]

    return pd.DataFrame(np.asarray(X_transformed), columns=feature_names)

def get_shap_explainer(
    pipeline: Pipeline,
    X_train: pd.DataFrame,
    model_name: str = "Model",
) -> shap.Explainer:
    """
    Create a SHAP explainer appropriate for the model type.

    Uses TreeExplainer for tree-based models (XGBoost, LightGBM)
    and KernelExplainer as a fallback for other model types.

    Args:
        pipeline: Fitted sklearn Pipeline (preprocessor + classifier).
        X_train: Training features (raw, before preprocessing).
        model_name: Model name for logging.

    Returns:
        SHAP Explainer instance.
    """
    # Extract the classifier from the pipeline
    classifier = pipeline.named_steps["classifier"]

    # Transform training data through every step except the classifier
    X_train_df = transform_features(pipeline, X_train)

    # Choose appropriate explainer
    model_type = type(classifier).__name__
    if model_type in ("XGBClassifier", "LGBMClassifier", "RandomForestClassifier"):
        logger.info(f"Using TreeExplainer for {model_name} ({model_type})")
        explainer = shap.TreeExplainer(classifier)
    elif hasattr(classifier, "coef_"):
        # Exact and fast for linear models (explains the log-odds margin).
        # KernelExplainer here would need ~2k model evals per explained row.
        logger.info(f"Using LinearExplainer for {model_name} ({model_type})")
        explainer = shap.LinearExplainer(classifier, X_train_df)
    else:
        logger.info(f"Using KernelExplainer for {model_name} ({model_type})")
        # Use a sample of training data for KernelExplainer (it's slow)
        background = shap.sample(X_train_df, min(100, len(X_train_df)))
        explainer = shap.KernelExplainer(classifier.predict_proba, background)

    return explainer

def compute_shap_values(
    pipeline: Pipeline,
    X: pd.DataFrame,
    explainer: shap.Explainer,
    max_samples: int = 500,
) -> tuple:
    """
    Compute SHAP values for a dataset.

    Args:
        pipeline: Fitted Pipeline.
        X: Raw features (before preprocessing).
        explainer: SHAP Explainer instance.
        max_samples: Maximum number of samples to explain (for speed).

    Returns:
        Tuple of (shap_values, X_transformed as DataFrame).
    """
    # Sample if dataset is large
    if len(X) > max_samples:
        X_sample = X.sample(n=max_samples, random_state=42)
    else:
        X_sample = X

    # Transform through feature engineering + preprocessing
    X_df = transform_features(pipeline, X_sample)

    # Compute SHAP values
    logger.info(f"Computing SHAP values for {len(X_df)} samples...")
    shap_values = explainer.shap_values(X_df)

    # For binary classification, explainers may return a list [class_0, class_1]
    # or a 3D array (n_samples, n_features, n_classes) — take the positive class
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    elif getattr(shap_values, "ndim", 2) == 3:
        shap_values = shap_values[:, :, 1]

    return shap_values, X_df

def plot_shap_summary(
    shap_values: np.ndarray,
    X_df: pd.DataFrame,
    model_name: str = "Model",
) -> None:
    """
    Create a SHAP beeswarm summary plot.

    Each point represents a single feature value for a single prediction.
    The x-axis shows the SHAP value (impact on prediction), and the
    color shows the feature value (red = high, blue = low).

    This is the most informative single visualization of model behavior.
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(
        shap_values, X_df,
        plot_type="dot",
        show=False,
        max_display=15,
    )
    plt.title(f"SHAP Summary — {model_name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(
        FIGURES_DIR / "shap_summary.png", dpi=150, bbox_inches="tight"
    )
    plt.close("all")
    logger.info("Saved SHAP summary plot (beeswarm).")

def plot_shap_bar(
    shap_values: np.ndarray,
    X_df: pd.DataFrame,
    model_name: str = "Model",
) -> None:
    """
    Create a SHAP bar plot showing mean absolute SHAP values.

    This shows which features have the largest impact on predictions
    on average, regardless of direction.
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(
        shap_values, X_df,
        plot_type="bar",
        show=False,
        max_display=15,
    )
    plt.title(f"Global Feature Importance (SHAP) — {model_name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(
        FIGURES_DIR / "shap_bar.png", dpi=150, bbox_inches="tight"
    )
    plt.close("all")
    logger.info("Saved SHAP bar plot (global importance).")

def plot_shap_waterfall(
    explainer: shap.Explainer,
    shap_values: np.ndarray,
    X_df: pd.DataFrame,
    sample_idx: int = 0,
    model_name: str = "Model",
) -> None:
    """
    Create a SHAP waterfall plot for a single prediction.

    Shows how each feature pushes the prediction from the base value
    (average prediction) to the final output. This is the best way
    to explain an individual prediction to stakeholders.
    """
    try:
        # Create an Explanation object
        if hasattr(explainer, "expected_value"):
            expected = explainer.expected_value
            if isinstance(expected, (list, np.ndarray)):
                expected = expected[1] if len(expected) > 1 else expected[0]
        else:
            expected = 0.0

        explanation = shap.Explanation(
            values=shap_values[sample_idx],
            base_values=expected,
            data=X_df.iloc[sample_idx].values,
            feature_names=X_df.columns.tolist(),
        )

        fig, ax = plt.subplots(figsize=(10, 8))
        shap.plots.waterfall(explanation, show=False, max_display=12)
        plt.title(
            f"SHAP Waterfall — Single Prediction ({model_name})",
            fontsize=13, fontweight="bold",
        )
        plt.tight_layout()
        plt.savefig(
            FIGURES_DIR / "shap_waterfall.png", dpi=150, bbox_inches="tight"
        )
        plt.close("all")
        logger.info("Saved SHAP waterfall plot.")
    except Exception as e:
        logger.warning(f"Could not create waterfall plot: {e}")

def plot_shap_dependence(
    shap_values: np.ndarray,
    X_df: pd.DataFrame,
    top_n: int = 3,
    model_name: str = "Model",
) -> None:
    """
    Create SHAP dependence plots for the top N most important features.

    Dependence plots show how the SHAP value of a feature changes as
    its value changes. The color shows the most interacting feature.
    """
    # Find top features by mean absolute SHAP value
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    top_indices = np.argsort(mean_abs_shap)[-top_n:][::-1]
    top_features = [X_df.columns[i] for i in top_indices]

    for feature in top_features:
        try:
            fig, ax = plt.subplots(figsize=(8, 5))
            shap.dependence_plot(
                feature, shap_values, X_df,
                show=False,
                ax=ax,
            )
            ax.set_title(
                f"SHAP Dependence — {feature} ({model_name})",
                fontsize=13, fontweight="bold",
            )
            plt.tight_layout()
            safe_name = feature.replace(" ", "_").replace("/", "_")
            fig.savefig(
                FIGURES_DIR / f"shap_dependence_{safe_name}.png",
                dpi=150, bbox_inches="tight",
            )
            plt.close(fig)
        except Exception as e:
            logger.warning(f"Could not create dependence plot for {feature}: {e}")

    logger.info(f"Saved SHAP dependence plots for top {top_n} features.")

def run_explainability(
    pipeline: Pipeline,
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    model_name: str = "Best Model",
) -> Dict[str, Any]:
    """
    Run the full SHAP explainability pipeline.

    Steps:
    1. Create SHAP explainer
    2. Compute SHAP values on test set
    3. Generate summary, bar, waterfall, and dependence plots

    Args:
        pipeline: Fitted best model Pipeline.
        X_train: Training features (for explainer background).
        X_test: Test features (to explain).
        model_name: Model name for plot titles.

    Returns:
        Dictionary with SHAP values and feature importance ranking.
    """
    logger.info("=" * 60)
    logger.info(f"SHAP EXPLAINABILITY — {model_name}")
    logger.info("=" * 60)

    # Create explainer
    explainer = get_shap_explainer(pipeline, X_train, model_name)

    # Compute SHAP values
    shap_values, X_df = compute_shap_values(pipeline, X_test, explainer)

    # Generate all plots
    plot_shap_summary(shap_values, X_df, model_name)
    plot_shap_bar(shap_values, X_df, model_name)
    plot_shap_waterfall(explainer, shap_values, X_df, sample_idx=0, model_name=model_name)
    plot_shap_dependence(shap_values, X_df, top_n=3, model_name=model_name)

    # Feature importance ranking
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    importance_df = pd.DataFrame({
        "feature": X_df.columns,
        "mean_abs_shap": mean_abs_shap,
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    logger.info("\nTop 10 features by SHAP importance:")
    for _, row in importance_df.head(10).iterrows():
        logger.info(f"  {row['feature']}: {row['mean_abs_shap']:.4f}")

    return {
        "shap_values": shap_values,
        "feature_importance": importance_df,
        "explainer": explainer,
    }

In [ ]:
shap_results = run_explainability(
    best_pipeline, X_train, X_test, model_name=best_model_name,
)

display(Image(filename=str(FIGURES_DIR / "shap_summary.png")))
display(Image(filename=str(FIGURES_DIR / "shap_bar.png")))
display(Image(filename=str(FIGURES_DIR / "shap_waterfall.png")))

## ✅ Wrap-Up

This notebook reproduced the full data-science pipeline — cleaning, EDA, feature
engineering, training, validation, comparison, threshold tuning, and explainability —
using code identical to the production scripts, and regenerated the same figures and
`reports/model_comparison.csv`.

**Two things intentionally stay in `python main.py`:**
- **Serving-artifact persistence** (`models/final_pipeline.joblib` & metadata) — pipelines
  pickled from a notebook reference `__main__`-defined classes, which the FastAPI service
  could not unpickle. The deployable artifact must come from the scripts.
- **MLflow experiment tracking** (`mlruns/`).

Serving stack: `uvicorn app.api:app` (REST API with the tuned threshold and automatic
model reload) and `streamlit run app/streamlit_app.py` (dashboard with SHAP waterfalls).